## Unified log viewer

**COMMAND:** `journalctl` — the single most-used tool in real support work

**PURPOSE (support):** Before `journalctl`, logs were scattered across separate files per service (`/var/log/cron.log`, `/var/log/syslog`, etc.) with inconsistent formats. `journalctl` is systemd's unified log viewer — every service managed by systemd (which is most of them now) logs through it, one consistent interface, filterable by service, time, and severity. This is your equivalent of tailing Django/Gunicorn error logs, except system-wide and structured.

**SYNTAX:** `journalctl [flags]`

**FLAGS:**

|Flag|Meaning|
|---|---|
|`-u UNIT`|logs for one specific service only (e.g. `-u cron`)|
|`-f`|follow — live tail, like `tail -f` but for the journal|
|`--since "TIME"`|only entries after this time|
|`--until "TIME"`|only entries before this time|
|`-p LEVEL`|filter by priority: `emerg, alert, crit, err, warning, notice, info, debug`|
|`-b`|only logs since last boot|
|`-r`|reverse order — newest first|
|`-n N`|show only the last N lines (like `tail -n`)|
|`--no-pager`|print directly, skip `less`|

**Developer bridge:** `journalctl -u cron -f` is your `tail -f gunicorn.log` equivalent. `-p err` is like filtering Django logs to `ERROR` level only, skipping `INFO`/`DEBUG` noise. `--since`/`--until` is exactly the time-window filtering you'd do querying a logs table or CloudWatch/Datadog — except local and instant.

**EXAMPLE:**

In [ ]:
$ journalctl -u cron --no-pager -n 10
Sep 08 16:10:44 HP systemd[1]: Started cron.service - Regular background program processing daemon.
Sep 08 16:10:44 HP cron[9821]: (CRON) INFO (pidfile fd = 3)
Sep 08 16:10:44 HP cron[9821]: (CRON) INFO (Running @reboot jobs)

**READ THE OUTPUT:** timestamp → hostname → unit/PID → message. Compare a healthy start (`Started cron.service...`) against a crash — you'd see `Main process exited, code=exited, status=1/FAILURE` right before systemd gives up and marks it `failed`.

**TRY NOW — run these one at a time, in order:**

**Command 1**:

In [ ]:
dennie@HP:~/linux-tut$ journalctl -u cron --no-pager -n 20
Sep 08 03:48:46 HP cron[159]: (CRON) INFO (pidfile fd = 3)
Sep 08 03:48:46 HP cron[159]: (CRON) INFO (Running @reboot jobs)
Sep 08 04:17:00 HP CRON[4964]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 04:17:00 HP CRON[4965]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 08 04:17:00 HP CRON[4964]: pam_unix(cron:session): session closed for user root
Sep 08 05:17:00 HP CRON[11169]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 05:17:00 HP CRON[11170]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 08 05:17:00 HP CRON[11169]: pam_unix(cron:session): session closed for user root
Sep 08 16:10:15 HP systemd[1]: Stopping cron.service - Regular background program processing daemon...
Sep 08 16:10:15 HP systemd[1]: cron.service: Deactivated successfully.
Sep 08 16:10:15 HP systemd[1]: Stopped cron.service - Regular background program processing daemon.
Sep 08 16:11:36 HP systemd[1]: Started cron.service - Regular background program processing daemon.
Sep 08 16:11:36 HP (cron)[14119]: cron.service: Referenced but unset environment variable evaluates to an empty string: EXTRA_OPTS
Sep 08 16:11:36 HP cron[14119]: (CRON) INFO (pidfile fd = 3)
Sep 08 16:11:36 HP cron[14119]: (CRON) INFO (Skipping @reboot jobs -- not system startup)
Sep 08 16:17:01 HP CRON[14205]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 16:17:01 HP CRON[14205]: pam_unix(cron:session): session closed for user root
Sep 08 18:17:00 HP CRON[14812]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 18:17:00 HP CRON[14813]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 08 18:17:00 HP CRON[14812]: pam_unix(cron:session): session closed for user root

Great real-world output — let's mine it, because this is exactly the skill support work tests: separating signal from noise in a live log stream.

**From your `-u cron -n 20` output:**

In [ ]:
Sep 08 16:11:36 HP (cron)[14119]: cron.service: Referenced but unset environment variable evaluates to an empty string: EXTRA_OPTS

This looks alarming ("unset variable!") but it's a well-known **benign** quirk in Debian/Ubuntu's cron systemd unit — `EXTRA_OPTS` is an optional config hook most installs never set, and cron logs a warning about it on every restart regardless. This is a great example of why `-p err` matters more than eyeballing raw text — this line is only priority `warning`, not `err`, so it correctly did **not** show up in your `-p err` query. That's the tool doing its job: separating "noisy but harmless" from "actually broken."

In [ ]:
Sep 08 16:11:36 HP cron[14119]: (CRON) INFO (Skipping @reboot jobs -- not system startup)

Confirms the earlier WSL point directly — this restart was **not** treated as a true system boot (`@reboot` jobs were correctly skipped), reinforcing that `-b` boundaries in WSL don't align with "I restarted the service."

**Command 2**:

In [ ]:
dennie@HP:~/linux-tut$ journalctl -u cron -p err --no-pager
-- No entries --

**`-p err` came back `-- No entries --`** — correct and expected. Cron is genuinely healthy. If a ticket said "cron is broken" and you ran this and got empty, that's a strong signal to look _elsewhere_ — the crontab entry itself, not the cron service.

**Command 3**:

In [ ]:
dennie@HP:~/linux-tut$ journalctl --since "10 minutes ago" --no-pager | tail -20
Sep 08 18:13:37 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:14:09 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:14:33 HP wsl-pro-service[14332]: WARNING Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory
Sep 08 18:14:41 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:15:12 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:15:32 HP wsl-pro-service[14332]: WARNING Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory
Sep 08 18:15:44 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:16:16 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:16:31 HP wsl-pro-service[14332]: WARNING Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory
Sep 08 18:16:42 HP PackageKit[14613]: daemon quit
Sep 08 18:16:42 HP systemd[1]: packagekit.service: Deactivated successfully.
Sep 08 18:16:47 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:17:00 HP CRON[14812]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 18:17:00 HP CRON[14813]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 08 18:17:00 HP CRON[14812]: pam_unix(cron:session): session closed for user root
Sep 08 18:17:19 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:17:30 HP wsl-pro-service[14332]: WARNING Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory
Sep 08 18:17:51 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:18:22 HP systemd-resolved[143]: Clock change detected. Flushing caches.
Sep 08 18:18:28 HP wsl-pro-service[14332]: WARNING Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory

**From `--since "10 minutes ago"`** — this one's a good lesson in scope. You asked for _all_ system logs (no `-u` filter), so you got the whole machine's noise, not just cron:

- `systemd-resolved: Clock change detected` repeating — WSL2's VM clock resyncing with the Windows host, typically after your laptop sleeps/wakes. Harmless, WSL-specific.
- `wsl-pro-service: could not connect to Windows Agent` — Ubuntu Pro service failing to find a file that doesn't exist on your setup. Also harmless noise, unrelated to anything you're doing.

The real diagnostic lesson: **`--since`/`--until` without `-u` is a firehose.** In an actual incident, you'd almost always combine them: `journalctl -u cron --since "13:55" --until "14:05"` — scoped by both service _and_ time, which is what the earlier ticket scenario spec

**Command 4**:

In [ ]:
dennie@HP:~/linux-tut$ journalctl -b --no-pager | wc -l
1019

**READ THE OUTPUT:** `-b` scopes to "since last boot," `--no-pager` prints it all directly instead of paging, and `wc -l` counts the total lines — so your WSL VM has logged **1,019 lines** since it last fully restarted. On its own this number isn't diagnostic — it's a baseline. Where it becomes useful:

- **Trending a problem:** if you check this count now, then again in an hour, and it's jumped to 50,000, something is logging excessively — a crash-looping service, a noisy driver, a process stuck in a restart loop. That's a real "why is the disk filling up" or "why is the system sluggish" lead.
- **Confirming the WSL boot-scope caveat from earlier:** 1,019 lines is a _lot_ for "since I opened this terminal tab" — it's further evidence that `-b` in WSL spans back to the last `wsl --shutdown`/Windows restart, not your current session, exactly as flagged. You can sanity-check the actual boundary with:

In [ ]:
dennie@HP:~/linux-tut$ journalctl -b --no-pager | head -5
Sep 08 03:48:45 HP kernel: Linux version 6.18.33.2-microsoft-standard-WSL2 (root@f1bbfb02316b) (gcc (GCC) 13.2.0, GNU ld (GNU Binutils) 2.41) #1 SMP PREEMPT_DYNAMIC Thu Jun 18 21:54:43 UTC 2026
Sep 08 03:48:45 HP kernel: Command line: initrd=\initrd.img WSL_ROOT_INIT=1 panic=-1 nr_cpus=4 hv_utils.timesync_implicit=1 console=hvc0 debug pty.legacy_count=0 WSL_ENABLE_CRASH_DUMP=1
Sep 08 03:48:45 HP kernel: KERNEL supported cpus:
Sep 08 03:48:45 HP kernel:   Intel GenuineIntel
Sep 08 03:48:45 HP kernel:   AMD AuthenticAMD

- and look at the very first timestamp — that's when this "boot" actually began.

---

#### Live-follow demo (this is the big one for incident response):

1st terminal:

In [ ]:
dennie@HP:~/linux-tut$ journalctl -u cron -f
Sep 08 16:10:15 HP systemd[1]: Stopped cron.service - Regular background program processing daemon.
Sep 08 16:11:36 HP systemd[1]: Started cron.service - Regular background program processing daemon.
Sep 08 16:11:36 HP (cron)[14119]: cron.service: Referenced but unset environment variable evaluates to an empty string: EXTRA_OPTS
Sep 08 16:11:36 HP cron[14119]: (CRON) INFO (pidfile fd = 3)
Sep 08 16:11:36 HP cron[14119]: (CRON) INFO (Skipping @reboot jobs -- not system startup)
Sep 08 16:17:01 HP CRON[14205]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 16:17:01 HP CRON[14205]: pam_unix(cron:session): session closed for user root
Sep 08 18:17:00 HP CRON[14812]: pam_unix(cron:session): session opened for user root(uid=0) by root(uid=0)
Sep 08 18:17:00 HP CRON[14813]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
Sep 08 18:17:00 HP CRON[14812]: pam_unix(cron:session): session closed for user root

2nd terminal:

In [ ]:
dennie@HP:~/linux-tut$ sudo systemctl restart cron
[sudo] password for dennie: 

Watch the first terminal — you should see the stop/start events appear in real time. Press `Ctrl+C` to exit the follow.

back to hanging 1st terminal:

In [ ]:
Sep 08 18:20:39 HP systemd[1]: Stopping cron.service - Regular background program processing daemon...
Sep 08 18:20:39 HP systemd[1]: cron.service: Deactivated successfully.
Sep 08 18:20:39 HP systemd[1]: Stopped cron.service - Regular background program processing daemon.
Sep 08 18:20:39 HP systemd[1]: Started cron.service - Regular background program processing daemon.
Sep 08 18:20:39 HP (cron)[15097]: cron.service: Referenced but unset environment variable evaluates to an empty string: EXTRA_OPTS
Sep 08 18:20:39 HP cron[15097]: (CRON) INFO (pidfile fd = 3)
Sep 08 18:20:39 HP cron[15097]: (CRON) INFO (Skipping @reboot jobs -- not system startup)
^C

That's a perfect live-follow capture — exactly what you'd watch during a real incident. Notice the sequence: `journalctl -u cron -f` sat idle showing history, then the instant you ran `restart` in the second terminal, five new lines streamed in _live_, in order: Stopping → Deactivated → Stopped → Started → the same benign `EXTRA_OPTS` warning on every restart. This is the core incident-response pattern — **one terminal tailing logs, a second terminal taking action** — and you'll use this constantly: watching `journalctl -f` while someone else (or you, in another tab) restarts a service, deploys code, or reproduces a bug.

One small thing to flag: `Ctrl+C` correctly exited the follow — that's the right way to break out of `-f` (unlike the pager `q` convention from `status`, since `-f` isn't a pager, it's a live stream, so `Ctrl+C` is the correct/only way to stop it).

You've now got the full `journalctl` toolkit down solid: scoping by unit, by priority, by time window, and live-following. That closes out `journalctl`.

---

That closes out all four `TRY NOW` commands plus the live-follow demo — `journalctl` is fully done.